In [0]:
# 1. CẤU HÌNH LIÊN KẾT CATALOG
CATALOG_NAME = "omni_retail_catalog"
spark.sql(f"USE CATALOG {CATALOG_NAME}")

silver_sales_table = f"{CATALOG_NAME}.silver.fnd_sales_items"
gold_fact_table = f"{CATALOG_NAME}.gold.fact_sales_daily"

print("Bắt đầu quy trình kiểm tra đối soát (Data Reconciliation Audit)...")

# 2. KIỂM THỬ ĐỐI SOÁT TỔNG DOANH THU THUẦN (FINANCIAL CHECKSUM)
from pyspark.sql import functions as F

silver_revenue = spark.table(silver_sales_table).select(
    F.sum("line_total")
).collect()[0][0]

gold_revenue = spark.table(gold_fact_table).select(
    F.sum("total_sales_amount")
).collect()[0][0]

diff_revenue = abs(float(silver_revenue) - float(gold_revenue))

# 3. KIỂM THỬ ĐỐI SOÁT TỔNG SỐ LƯỢNG HÀNG BÁN (QUANTITY CHECKSUM)
silver_units = spark.table(silver_sales_table).select(
    F.sum("quantity")
).collect()[0][0]

gold_units = spark.table(gold_fact_table).select(
    F.sum("quantity")
).collect()[0][0]

diff_units = abs(int(silver_units) - int(gold_units))

# 4. KIỂM THỬ ĐỐI SOÁT TỔNG SỐ ĐƠN HÀNG DUY NHẤT (DISTINCT ORDER COUNT)
silver_orders = spark.table(silver_sales_table).select(
    F.countDistinct("order_id")
).collect()[0][0]

gold_orders = spark.table(gold_fact_table).select(
    F.countDistinct("order_id")    # <-- ĐÃ SỬA: Đếm distinct order_id chuẩn xác
).collect()[0][0]

diff_orders = abs(int(silver_orders) - int(gold_orders))

# 5. XUẤT BẢNG BÁO CÁO NGHIỆM THU (AUDIT REPORT)
print("=" * 60)
print("           BÁO CÁO ĐỐI SOÁT DỮ LIỆU TỰ ĐỘNG (UAT SIGN-OFF)")
print("=" * 60)
print(f"1. Tổng doanh thu (Revenue Checksum):")
print(f"   - Tầng Silver : ${float(silver_revenue):,.2f}")
print(f"   - Tầng Gold   : ${float(gold_revenue):,.2f}")
print(f"   - Chênh lệch  : ${diff_revenue:,.4f}")

print(f"\n2. Tổng số lượng sản phẩm (Units Sold):")
print(f"   - Tầng Silver : {silver_units:,}")
print(f"   - Tầng Gold   : {gold_units:,}")
print(f"   - Chênh lệch  : {diff_units}")

print(f"\n3. Tổng số lượt đơn hàng (Orders):")
print(f"   - Tầng Silver : {silver_orders:,}")
print(f"   - Tầng Gold   : {gold_orders:,}")
print(f"   - Chênh lệch  : {diff_orders}")
print("=" * 60)

assert diff_revenue < 0.01, f"LỖI: Sai lệch doanh thu! Silver: {silver_revenue}, Gold: {gold_revenue}"
assert diff_units == 0, f"LỖI: Sai lệch số lượng hàng bán! Silver: {silver_units}, Gold: {gold_units}"
assert diff_orders == 0, f"LỖI: Sai lệch số lượng đơn hàng! Silver: {silver_orders}, Gold: {gold_orders}"

print(">>> XÁC NHẬN: 100% SỐ LIỆU ĐỐI SOÁT HOÀN TOÀN CHÍNH XÁC (AUDIT PASSED) <<<")